# Demo & Exercise Kalman Filter

We would like to explore the Kalman filter a little bit more with some exercises.


In [ ]:
import numpy as np
# for displaying images in jupyter
import matplotlib as mpl
from matplotlib import pyplot as plt
%matplotlib inline
mpl.rcParams['figure.dpi']= 200

## State vector
First we would like to track an object that would be thrown and therefor moves according to a kinematic model and the gravitation of the earth.

The state needs to be the position and the velocity of the object, we can also add the acceleration due to gravity, as this will simplify the matrices. 

The system model describes how the object moves and the measurement model describes the relation between the measurement and the state. We assume that we can only measure the position, but not the velocity.


In [ ]:
# state
position = [0, 0]
velocity = [10, 20]
acceleration = [0, -9.81]

# concatenate lists and store as numpy array, we use the variable x for the whole state
x0 = np.asarray(position + velocity + acceleration)
print(x0)

## System Model

Next we will need to derive the system model. We know that the next position is the previous position plus how much has been moved according to the velocity, similarly the velocity changes linearly with the acceleration, and we assume that the acceleration stays the same, i.e.

$$
\begin{align}
p_{t+1} &= p_t + \Delta t \cdot v_t\\
v_{t+1} &= v_t + \Delta t \cdot a_t\\
a_{t+1} &= a_t\\
\end{align}
$$

We need to capture this as a matrix so that the state vector at time t+1: $x_{t+1}$ can be calculated by applying the system model $A$:
$$
x_{t+1} = A  x_{t}
$$

As the state has 6 values, so the matrix will be a 6x6 matrix. The diagonal elements are all 1, set the remaining non-zero values, so that the matrix $A$ describes the equations for $p, v$ and $a$ above.

In [ ]:
A = np.diag([1., 1., 1., 1., 1., 1.])
deltaT = 0.1

# YOUR CODE HERE
raise NotImplementedError()


In [ ]:
x_new = np.matmul(A, x0)
print(x_new)
np.testing.assert_array_equal(x_new, [1.,2.,10.,19.019,0.,-9.81])


If we continue to apply the system model, we will get all the states of the object over time.

In [ ]:
steps = 50

# Let us store the resulting state vector in an array x, we star
x = np.zeros(shape=(steps, x0.shape[0]))
x[0,:] = x0
for i in range(1,steps):
    x[i,:] = np.matmul(A, x[i-1,:])

In [ ]:
fig, ax = plt.subplots(figsize=(12,8))
ax.plot(x[:,0], x[:,1], label='Position')
ax.legend()

We have seen in the lecture, that the Kalman filter uses a noise model. Why is that? 

First of all, if we would have a model that is absolutely correct and we also know the initiale state accurately, then we could just calculate all the remaining positions. However, we might not know the model or the model might not be correct, for example we have neclected the air resistance. Also we might have an error in the estimation of the initial state, so the next states are less certain.

The uncertainity of the system is modelled as a matrix Q. We will assume that the matrix is diagonal, so that state and velocity noises are independant.

In [ ]:
# System noise model
sigmaSystemPos = 0.2
sigmaSystemVel = 0.2
Q = np.zeros(shape=(6,6));
Q[0,0] = sigmaSystemPos;
Q[1,1] = sigmaSystemPos;
Q[2,2] = sigmaSystemVel;
Q[3,3] = sigmaSystemVel;
print(Q)

So we will also have a uncertainity in our estimation. The current value of the uncertainity is also modelled by a matrix. In the equations, this is called P.

In [ ]:
P = np.zeros(shape=(6,6));

In the linear Kalman filter, P evolves according to 
$$
P_{t+1} = A P_t A^T + Q
$$

We can calculate this and plot the uncertainity.

In [ ]:
x = np.zeros(shape=(steps, x0.shape[0]))

p = np.zeros(shape=(steps,x0.shape[0],x0.shape[0]))
x[0,:] = x0
p[0,:] = P
             
        
for i in range(1,steps):
    x[i,:] = np.matmul(A, x[i-1,:])
    p[i,:] = np.matmul(A, np.matmul(p[i-1,:], np.transpose(A))) + Q

In [ ]:
# we will only plot the error in the y value of the position, there is a similar error in the x position. 
# The value in the matrix is the square of the error, so we will take the square root.

fig, ax = plt.subplots(figsize=(12,8))
ax.errorbar(x[:,0], x[:,1], yerr=np.sqrt(p[:,1,1]),label='Position')
ax.legend()

As we can see, the error or uncertainity gets larger over time.

## Measurements

We will assume that we can only measure the position of the object, but not its velocity or acceleration. In the Kalman filter, that must be again specified by a matrx $C$ that calculate the measurement $z$ from the state $x$:
$$
z = C x
$$

Fill the matrix C with the correct value. In order to make this easier, the dimension of C is already given.

In [ ]:
# Measurement Model
C = np.zeros(shape=(2,6));
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
assert C.sum() == 2

The measurement might not be exact either, this is again modelled by a matrix which in that case is a 2x2 matrix R.


In [ ]:
# Measurement noise
sigmaMeasureX = 0.1
sigmaMeasureY = 0.5
R = np.zeros(shape=(2,2));
R[0,0] = sigmaMeasureX;
R[1,1] = sigmaMeasureY;
print(R)

This measurement noise is the one that we would *expect* and that will be used by the Kalman filter to model the uncertainity.

Of course it should be similar to the actual measurement noise. So we will use a similar distribution. In order to simulate measurements, we will just take the true state values and corrupt them by noise.

In [ ]:
sigmaMeasureXExperiment = 0.1
sigmaMeasureYExperiment = 0.5
z = np.zeros(shape=(steps, 2))
for i in range(0,steps):
    z[i,0] = np.random.normal(loc=x[i,0], scale=sigmaMeasureXExperiment)
    z[i,1] = np.random.normal(loc=x[i,1], scale=sigmaMeasureYExperiment)

We can know plot the measurements together with the true values.

In [ ]:
fig, ax = plt.subplots(figsize=(12,8))
ax.plot(x[:,0], x[:,1], label='Position')
ax.plot(z[:,0], z[:,1], marker='x', linestyle="None", label='Measurement')
ax.legend()

## Kalman filter

So, we can finally apply the Kalman filter to not only get better expectations of the position, but we could also get the other values of the state.

We will *not* implement the Kalman filter ourselves, but use the implementation below :-).

In [ ]:
def kalman_step(A, C, Q, R, z, x, V, initial):
    """
    Args:
    A: the system matrix
    C: the observation matrix 
    Q: the system covariance 
    R: the observation covariance
    z: the observation at time t
    x: prior mean
    V: - prior covariance
    initial: true if this is an initial step (means x and V are taken as initial conditions (so A and Q are ignored)
    
    Returns:
      x_new: posterior mean
      V_new: posterior covariance

    """
    if initial:
        x_pred = x
        V_pred = V
    else:
        x_pred = np.matmul(A, x)
        V_pred = np.matmul(A, np.matmul(V, np.transpose(A))) + Q
        
    e = z - np.matmul(C, x_pred);
    C_trans = np.transpose(C)
    S = np.matmul(V_pred,C_trans)
    S = np.matmul(C, S) + R
    S_inverse = np.linalg.inv(S)

    # Kalman gain matrix
    K = np.matmul(np.matmul(V_pred, C_trans), S_inverse)
    
    x_res = x_pred + np.matmul(K, e)
    V_res = np.identity(V.shape[0]) - np.matmul(K, C)
    V_res = np.matmul(V_res, V_pred)
    
    return x_res, V_res

Let us apply the filter to calculate the state and the uncertainity at each time step:

In [ ]:
steps = 50
# estimated state values
x_est = np.zeros(shape=(steps, x0.shape[0]))
# estimated uncertainity
P_est = np.zeros(shape=(steps,x0.shape[0],x0.shape[0]))
x_est[0,:], P_est[0,:] = kalman_step(A, C, Q, R, z[0,:], x[0,:],p[0,:], initial=True)
for i in range(1,steps):
    x_est[i,:], P_est[i,:] = kalman_step(A, C, Q, R, z[i,:], x_est[i-1,:],P_est[i-1,:], initial=False)

In [ ]:
fig, ax = plt.subplots(figsize=(12,8))
ax.plot(x[:,0], x[:,1], label='Position')
ax.plot(z[:,0], z[:,1], marker='x', linestyle="None", label='Measurement')
ax.plot(x_est[:,0], x_est[:,1], marker='o', linestyle="None", label='Prediction')
ax.legend()

We can also add the error bars again (at the estimated positions)

In [ ]:
fig, ax = plt.subplots(figsize=(12,8))
ax.plot(x[:,0], x[:,1], label='Position')
ax.errorbar(x_est[:,0], x_est[:,1], yerr=np.sqrt(P_est[:,1,1]), marker='o', linestyle="None", label='Prediction')
ax.legend()